In [1]:
import pandas as pd
import numpy as np
import spacy
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer, util
import json

c:\Users\saima\anaconda3\envs\project_2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("data/users.csv")
df.location.value_counts()

location
Noida         13
Kolkata        9
Delhi          8
Kochi          8
Bangalore      8
Chandigarh     7
Chennai        7
Mumbai         7
Mysuru         6
Hyderabad      5
Ahmedabad      5
Jaipur         5
Gurugram       4
Pune           4
Coimbatore     4
Name: count, dtype: int64

In [6]:
nlp = spacy.load('en_core_web_sm')

In [7]:
df['combined_text'] = df['about_me'] + " " + df['professional_summary'] + " " + df['interests'] + " " + df["profession"]


In [8]:
clean_text = []
for i in df.combined_text:
    doc = nlp(i)
    line = [word.lemma_.lower() for word in doc if not word.is_punct and not word.is_stop]
    clean_text.append(' '.join(line))

In [9]:
clean_text

['passionate make difference work believe power collaboration clear communication personal time spend psychology marathon run look partner share drive impact authenticity result orient devops engineer 2 year cloud infrastructure strong background aws monitoring look collaborate project streamline operation efficiency growth psychology marathon running mentoring devops engineer',
 'organized detail orient professional value big picture thinking outside work deeply interested open source philosophy appreciate people reliable thoughtful eager grow experience marketing specialist 9 + year help healthcare organization achieve process automation skilled email campaigns google analytics content marketing currently focus build resilient secure system e commerce space open source philosophy machine learning marketing specialist',
 'organized detail orient professional value big picture thinking outside work deeply interested cycling robotic appreciate people reliable thoughtful eager grow exper

In [10]:
vectorizer = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3169.53it/s]


In [15]:
combined_vector = vectorizer.encode(clean_text)
np.save('data/user_vectors.npy', combined_vector)

In [18]:
combined_vector.shape

(100, 384)

In [2]:
from matching_functions import get_top_matches

test_user = {
    'professional_summary': 'Data scientist with 3 years experience in ML and Python',
    'about_me': 'I love solving problems and mentoring others',
    'interests': 'Chess, Reading, Machine Learning',
    'profession': 'Data Scientist',
    'mbti': 'INTP',
    'location': 'Bangalore'
}

weights = {'w1': 0.5, 'w2': 0.3, 'w3': 0.2}
matches = get_top_matches(test_user, weights)
print(matches)

C:\Users\saima\anaconda3\envs\project_2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 ... (more hidden) ...


   user_id              name            profession   location  mbti  \
45    U046   Revathi Nambiar  Marketing Specialist  Bangalore  ENTP   
76    U077     Harish Pandey    Research Scientist  Bangalore  INFJ   
53    U054    Divya Krishnan          Scrum Master  Bangalore  ENTJ   
15    U016      Vikram Patel      Business Analyst  Bangalore  ENTJ   
77    U078  Bhavna Choudhary  Full Stack Developer  Bangalore  ISTP   

                                 professional_summary  nlp_score  mbti_score  \
45  Results-oriented Marketing Specialist with 9 y...      0.381         0.9   
76  Research Scientist with 8 years of experience ...      0.595         0.4   
53  Scrum Master at the intersection of agile deli...      0.554         0.4   
15  Business Analyst with 12 years of experience i...      0.515         0.4   
77  Experienced Full Stack Developer with 9+ years...      0.498         0.4   

    location_score  total_score  
45             1.0        0.660  
76             1.0      

In [3]:
matches

,user_id,name,profession,location,mbti,professional_summary,nlp_score,mbti_score,location_score,total_score
45,U046,Revathi Nambiar,Marketing Specialist,Bangalore,ENTP,Results-oriented Marketing Specialist with 9 y...,0.381,0.9,1.0,0.660
76,U077,Harish Pandey,Research Scientist,Bangalore,INFJ,Research Scientist with 8 years of experience ...,0.595,0.4,1.0,0.617
53,U054,Divya Krishnan,Scrum Master,Bangalore,ENTJ,Scrum Master at the intersection of agile deli...,0.554,0.4,1.0,0.597
15,U016,Vikram Patel,Business Analyst,Bangalore,ENTJ,Business Analyst with 12 years of experience i...,0.515,0.4,1.0,0.577
77,U078,Bhavna Choudhary,Full Stack Developer,Bangalore,ISTP,Experienced Full Stack Developer with 9+ years...,0.498,0.4,1.0,0.569
